# Oracle Hybrid Search Mode

This notebook demonstrates `retrieval_mode="hybrid_search"`.

What you should see:

1. A seed run calls Tavily and writes results into Oracle.
2. A local-only check proves Oracle can retrieve the persisted rows.
3. A hybrid run returns a mix of Oracle local rows and fresh Tavily rows.
4. The final cell deletes this notebook's demo rows so the notebook can be rerun cleanly.

## Setup

Run this cell first. It loads `.env`, connects to Oracle, prepares the demo table, and imports shared notebook helpers.

In [ ]:
from pathlib import Path
import sys

helper_path = None
for root in [Path.cwd(), *Path.cwd().parents]:
    candidate = root / "examples" / "oracle" / "demo_helpers.py"
    if candidate.exists():
        helper_path = candidate
        break

if helper_path is None:
    candidate = Path.cwd() / "demo_helpers.py"
    if candidate.exists():
        helper_path = candidate

if helper_path is None:
    raise RuntimeError("Could not find examples/oracle/demo_helpers.py. Start Jupyter from the repository root or examples/oracle.")

sys.path.insert(0, str(helper_path.parent))
from demo_helpers import *

print("Using helper:", helper_path)

## Choose the demo query

Edit only `HYBRID_QUERY` below when you want to try your own query. The cleanup cells use this same value, so reruns stay predictable.

In [ ]:
HYBRID_QUERY = "Oracle Database vector search for Tavily hybrid retrieval"

## Start from a clean query-specific state

This removes only rows for `HYBRID_QUERY`, not the whole table.

In [ ]:
delete_rows_for_query(HYBRID_QUERY)

## 1. Seed Oracle with Tavily results

`max_local=0` forces this first call to use Tavily only. With `save_foreign=True`, those Tavily results are written into Oracle.

In [ ]:
hybrid_client = make_client(
    "hybrid_search",
    persistence_depth="cache_plus_memory",
)

seed_results = hybrid_client.search(
    HYBRID_QUERY,
    max_results=3,
    max_local=0,
    max_foreign=3,
    save_foreign=True,
    **TAVILY_SEARCH_OPTIONS,
)

show_results("Seed run: Tavily results persisted into Oracle", seed_results)
show_persisted_rows(HYBRID_QUERY, "Oracle rows after seed run")

## 2. Prove Oracle can retrieve the saved rows

`max_foreign=0` prevents Tavily from being called. If results appear here, they came from Oracle.

In [ ]:
local_results = hybrid_client.search(
    HYBRID_QUERY,
    max_results=3,
    max_local=3,
    max_foreign=0,
    save_foreign=False,
)

show_results("Oracle local-only check", local_results)

## 3. Run true hybrid search

This call allows both Oracle local rows and Tavily fresh rows, then ranks the combined set.

In [ ]:
hybrid_results = hybrid_client.search(
    HYBRID_QUERY,
    max_results=5,
    max_local=3,
    max_foreign=3,
    save_foreign=True,
    **TAVILY_SEARCH_OPTIONS,
)

show_results("Hybrid search: Oracle + Tavily", hybrid_results)
show_persisted_rows(HYBRID_QUERY, "Oracle rows after hybrid run")

## Cleanup

Run this at the end. It deletes this notebook's demo rows so the next full run starts with a Tavily seed again.

In [ ]:
delete_rows_for_query(HYBRID_QUERY)
print("Cleaned hybrid-search demo rows. Re-run from the top to see the first call use Tavily again.")